# Decision-payoff selection vs. statistical accuracy

A simple binary-gamble example to make one point: **the model that gives the best decision depends on what features of the distribution the agent cares about.** Agents with low risk aversion mostly care about the mean of the payoff; agents with high risk aversion mostly care about the severity of the left tail. So three agents with different $\gamma$ can rationally select different models from the same set of candidates, even on the same data and the same procedure.

**The true gamble.** Each period the payoff multiple is $M_+ = 1.5$ with true probability $p = 0.55$, otherwise $M_- = 0.7$. Per-period wealth evolves as
$$
W_{t+1} = W_t \cdot \bigl[1 + f(M - 1)\bigr], \qquad M \in \{M_+, M_-\}.
$$
The true mean payoff is $0.55 \cdot 1.5 + 0.45 \cdot 0.7 = 1.14$.

**Three candidate models.** All three are misspecified, none matches the truth. But by construction **all three agree with the truth on the mean payoff (1.14)**; they only disagree on how severe the left tail is — i.e., on how bad the loss state $M_-$ is.

| Model | $q$ | $M_+$ | $M_-$ | character |
|---|---:|---:|---:|---|
| A — thin left tail | 0.31 | 1.897 | 0.80 | rare big wins, mild losses |
| B — medium left tail | 0.76 | 1.326 | 0.55 | frequent small wins, medium losses |
| C — fat left tail | 0.88 | 1.255 | 0.30 | almost-always tiny wins, occasional severe loss |

**Three agents** with CRRA risk aversion $\gamma \in \{0.5, 1, 2\}$. Each agent considers all three models, picks an $f^\star$ under each, and selects among them by realized mean utility on a 1000-draw validation set from the true gamble.

**What we expect, and why.** A low-$\gamma$ agent is approximately mean-sensitive: she likes leverage and is forgiving of losses. Since all three models agree on the mean, what distinguishes them in her eyes is whether their recommended $f^\star$ is *appropriately aggressive* for someone who cares mostly about the mean. The model with the thinnest perceived tail (A) recommends the most aggressive $f^\star$, matching her taste. A high-$\gamma$ agent is loss-averse: she fears the deep left tail. The model that *takes the left tail most seriously* (C) recommends the most conservative $f^\star$, matching her caution. The medium agent splits the difference and prefers B.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

rng = np.random.default_rng(42)

p_true, Mp_true, Mm_true = 0.55, 1.5, 0.7
models = {
    "A (thin left tail)":   {"q": 0.31, "M_plus": 1.897, "M_minus": 0.80},
    "B (medium left tail)": {"q": 0.76, "M_plus": 1.326, "M_minus": 0.55},
    "C (fat left tail)":    {"q": 0.88, "M_plus": 1.255, "M_minus": 0.30},
}
gammas = [0.5, 1.0, 2.0]

# Verify all candidates agree with truth on the mean
print(f"truth mean: {p_true*Mp_true + (1-p_true)*Mm_true:.4f}")
for name, m in models.items():
    print(f"{name}: mean = {m['q']*m['M_plus'] + (1-m['q'])*m['M_minus']:.4f}")

truth mean: 1.1400
A (thin left tail): mean = 1.1401
B (medium left tail): mean = 1.1398
C (fat left tail): mean = 1.1404


## Training stage: $f^\star$ under each candidate model

Each agent computes
$$
f^\star(\gamma, \text{model}) = \arg\max_f \; \mathbb{E}_{\text{model}}\bigl[u_\gamma(1 + f(M - 1))\bigr]
$$
with the expectation taken under the model's own beliefs $(q, M_+, M_-)$. We also report the truth-implied $f^\star(\gamma)$ as a benchmark — the $f^\star$ an agent who knew the truth would choose.

In [2]:
def neg_eu(f, q, Mp, Mm, gamma):
    wW = 1.0 + f * (Mp - 1.0)
    wL = 1.0 + f * (Mm - 1.0)
    if wW <= 0 or wL <= 0:
        return 1e10
    if gamma == 1:
        return -(q * np.log(wW) + (1 - q) * np.log(wL))
    return -(q * (wW ** (1 - gamma) - 1) / (1 - gamma)
             + (1 - q) * (wL ** (1 - gamma) - 1) / (1 - gamma))

def f_star(q, Mp, Mm, gamma, f_max=5.0):
    f_ruin = 1.0 / (1 - Mm) if Mm < 1 else f_max
    upper = min(f_ruin - 1e-4, f_max)
    res = minimize_scalar(neg_eu, args=(q, Mp, Mm, gamma),
                          bounds=(0.0, upper), method="bounded",
                          options={"xatol": 1e-7})
    return res.x

f_truth = np.array([f_star(p_true, Mp_true, Mm_true, g) for g in gammas])
fstar_table = pd.DataFrame(
    {name: [f_star(m["q"], m["M_plus"], m["M_minus"], g) for g in gammas]
     for name, m in models.items()},
    index=pd.Index(gammas, name="gamma"),
)
fstar_table.insert(0, "truth f*", f_truth)
fstar_table.round(3)

,truth f*,A (thin left tail),B (medium left tail),C (fat left tail)
gamma,,,,
0.5,1.805,1.791,1.582,1.169
1.0,0.933,0.781,0.953,0.787
2.0,0.460,0.355,0.511,0.453


Notice the pattern. As $\gamma$ rises (left to right within a model's row), every model recommends less leverage. But the *cross-model* ordering is preserved: at every $\gamma$, A recommends the most aggressive $f^\star$ and C recommends the most conservative one. That's because A has the mildest perceived loss state and C has the worst. Now look at how the truth-implied $f^\star$ compares across $\gamma$ to the three candidate columns:

- At $\gamma = 0.5$: truth = 1.805. **A is closest** (1.79).
- At $\gamma = 1$: truth = 0.933. **B is closest** (0.95).
- At $\gamma = 2$: truth = 0.460. **C is closest** (0.45).

This is the formal version of "different agents care about different things." The low-$\gamma$ agent's truth-implied $f^\star$ happens to coincide with A's recommendation; the high-$\gamma$ agent's coincides with C's. None of the candidate models is the truth, but each is *aligned with the truth for an agent of a particular risk aversion*.

## Validation stage: 1000 draws from the truth

Each $(\gamma, \text{model})$ pair's realized mean utility on 1000 i.i.d. realizations of the true gamble:
$$
\frac{1}{n} \sum_{t} u_\gamma\bigl(1 + f^\star_{\gamma, \text{model}}\,(M_t - 1)\bigr).
$$
The winning model per $\gamma$ is the one whose recommended $f^\star$ delivers the highest realized utility on data drawn from the true process.

In [3]:
n_draws = 1000
wins = rng.random(n_draws) < p_true
payoffs = np.where(wins, Mp_true, Mm_true)
print(f"realized win frequency: {wins.mean():.3f}  (truth p = {p_true})")

def realized_mean_utility(f, payoffs, gamma):
    w = 1.0 + f * (payoffs - 1.0)
    if np.any(w <= 0):
        return -np.inf
    if gamma == 1:
        return np.log(w).mean()
    return ((w ** (1 - gamma) - 1) / (1 - gamma)).mean()

val_table = pd.DataFrame(
    {name: [realized_mean_utility(fstar_table.loc[g, name], payoffs, g) for g in gammas]
     for name in models},
    index=pd.Index(gammas, name="gamma"),
)
winners = val_table.idxmax(axis=1).rename("winner")
pd.concat([val_table.round(5), winners], axis=1)

realized win frequency: 0.545  (truth p = 0.55)


,A (thin left tail),B (medium left tail),C (fat left tail),winner
gamma,,,,
0.5,0.11968,0.11834,0.10532,A (thin left tail)
1.0,0.05819,0.05916,0.05829,B (medium left tail)
2.0,0.02793,0.02853,0.02909,C (fat left tail)


## What we just saw

Each $\gamma$ picked a different model:

- **$\gamma = 0.5$ picks A (the thin-tail model).** A low-$\gamma$ agent's CRRA utility weights the *mean* of the payoff heavily relative to the tail; the linear (first-order) component of $u_\gamma$ dominates near $w = 1$. Since all three candidates agree with the truth on the mean, the agent's job is to pick the candidate whose recommended $f^\star$ is aggressive enough to capitalize on that mean. A — which thinks the loss state is mild and so recommends aggressive leverage — wins. The other two candidates *underestimate* the leverage a mean-focused agent would want.

- **$\gamma = 2$ picks C (the fat-tail model).** A high-$\gamma$ agent's CRRA utility is dominated by the loss state: $u_\gamma(W_t \cdot M_-)$ contributes a large negative number whenever the loss happens, and that pain is what the agent is mostly trying to avoid. C is the candidate that *takes the loss state most seriously*; its $f^\star$ recommendation is conservative enough to keep the realized loss bearable. The other two candidates *understate* the loss-state damage and so recommend too much leverage for a loss-averse agent.

- **$\gamma = 1$ (log utility, the Kelly agent) picks B (the medium-tail model).** The Kelly agent is in the middle — mean-sensitive but also penalized by losses through the log. B's $f^\star$ recommendation is the one that splits the difference.

**The pedagogical punchline.** Statistical accuracy is not the right model-selection criterion when a decision is downstream. Each of our three candidate models is correct about the mean of the gamble but wrong about its tail. Whether being right about the mean or right about the tail is more important depends on what the agent is going to do with the model. Two analysts with the same data and the same candidate models can rationally arrive at different choices because they care about different decision losses.

**Robustness.** At $n = 1000$ there is real sampling noise in the realized win frequency $\hat p$, so on some seeds the winners shuffle around. Asymptotically (try $n = 100{,}000$) the canonical pattern A→B→C across the three $\gamma$ is the deterministic outcome — see the $f^\star$ table, which is fixed before we draw any data.

**Connection to notebook 2.** Same principle as `06_Portfolio_optimization_and_copulas/2_simple_portfolio_optimization.ipynb`: there the candidates are continuous return distributions (Normal, Student-$t$, Johnson $S_U$), and the validation-stage winner depends on $\gamma$ because those distributions also differ in their tail behavior. Here the candidates are binary gambles; the structural lesson — agents at different $\gamma$ rationally pick different models — transfers.